# Experiment A — Calibration & Uncertainty
**Toward Robust Deep Learning: Evaluating Uncertainty in Transformer-Based Architectures for Scientific Data**

Nada Elhassan · University of Messina · ICTP July 2026

---
### What this notebook does
1. Loads the **MAGIC Gamma Telescope** dataset (astroparticle physics)
2. Trains a **Transformer** on it
3. Shows it is **overconfident** (standard baseline)
4. Fixes it with **MC Dropout** [Gal & Ghahramani 2016]
5. Fixes it with **Temperature Scaling** [Guo et al. 2017]
6. Generates **5 poster-quality figures**

### Papers to read alongside this notebook
| Step | Paper | Link |
|------|-------|------|
| Model | Vaswani et al. 2017 — *Attention is All You Need* | https://arxiv.org/abs/1706.03762 |
| MC Dropout | Gal & Ghahramani 2016 — *Dropout as a Bayesian Approximation* | https://arxiv.org/abs/1506.02142 |
| ECE + Temp. Scaling | Guo et al. 2017 — *On Calibration of Modern Neural Networks* | https://arxiv.org/abs/1706.04599 |
| OOD / Uncertainty | Ovadia et al. 2019 — *Can You Trust Your Model's Uncertainty?* | https://arxiv.org/abs/1906.02530 |

In [ ]:
# ── CELL 1: Install & clone ──────────────────────────────────────────────────
!pip install netcal --quiet
!git clone https://github.com/nadaibrahim-png/ictp-poster-2026.git 2>/dev/null || \
 (cd ictp-poster-2026 && git pull)
%cd ictp-poster-2026
print('✓ Ready')

In [ ]:
# ── CELL 2: Config check — tweak here if needed ──────────────────────────────
import config

config.DATASET  = 'magic'   # 'magic' | 'higgs' | 'wine'
config.EPOCHS   = 80
config.MC_SAMPLES = 30
config.OOD_NOISE  = 3.0

print(f'Dataset  : {config.DATASET}')
print(f'Epochs   : {config.EPOCHS}')
print(f'MC passes: {config.MC_SAMPLES}')
print(f'OOD noise: σ={config.OOD_NOISE}')

In [ ]:
# ── CELL 3: Load & preprocess data ───────────────────────────────────────────
# PAPER: Guo 2017 — 'We evaluate calibration on a held-out test set.'
# Split: 60% train | 20% val (for Temperature Scaling only) | 20% test
import numpy as np, torch
torch.manual_seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)

from data import get_dataset, preprocess, make_ood, get_loaders

X, y, n_classes, dataset_name, feature_names = get_dataset()
(X_train, y_train), (X_val, y_val), (X_test, y_test), scaler = preprocess(X, y)
X_ood   = make_ood(X_test)
loaders = get_loaders(X_train, y_train, X_val, y_val, X_test, y_test, X_ood)

print(f'\nDataset  : {dataset_name}')
print(f'Features : {X_train.shape[1]}  |  Classes: {n_classes}')
print(f'Train    : {X_train.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_test.shape[0]}')

In [ ]:
# ── CELL 4: Build & train the Transformer ────────────────────────────────────
# PAPER: Vaswani 2017 — self-attention across feature tokens
# PAPER: Gal 2016    — MC Dropout layer added before classifier head
from models   import TabTransformer
from training import train

model = TabTransformer(n_features=X_train.shape[1], n_classes=n_classes)
print(f'Parameters: {model.count_parameters():,}')

history = train(model, loaders)

In [ ]:
# ── CELL 5: Standard inference (BASELINE) ───────────────────────────────────
# Dropout OFF — single forward pass — the 'default' way models are deployed.
# PAPER: Guo 2017 Section 3 — 'Modern NNs are poorly calibrated.
#        They are overconfident even when wrong.'
# Expected: HIGH ECE (confidence >> accuracy)
from evaluation import predict_standard, compute_all

probs_std, labels_test = predict_standard(model, loaders['test'])
metrics_std = compute_all(probs_std, labels_test)

print('── Standard Baseline ──')
print(f"Accuracy : {metrics_std['accuracy']:.4f}")
print(f"ECE      : {metrics_std['ece']:.4f}   ← should be high (overconfident)")
print(f"NLL      : {metrics_std['nll']:.4f}")

In [ ]:
# ── CELL 6: MC Dropout ───────────────────────────────────────────────────────
# PAPER: Gal & Ghahramani 2016 Algorithm 1:
#   'At test time, keep dropout ON. Run T forward passes.
#    Predictive mean = (1/T) Σ softmax(f_dropout(x))'
#
# How it works in code: model.enable_mc_dropout() sets ONLY the mc_dropout
# layer to train() — everything else stays eval().
# The variance across 30 passes = epistemic uncertainty.
from evaluation import predict_mc_dropout

probs_mc,  _, uncertainty_test = predict_mc_dropout(model, loaders['test'])
probs_mc_ood, _, uncertainty_ood = predict_mc_dropout(model, loaders['ood'])
metrics_mc = compute_all(probs_mc, labels_test)

print('── MC Dropout (30 passes) ──')
print(f"Accuracy : {metrics_mc['accuracy']:.4f}")
print(f"ECE      : {metrics_mc['ece']:.4f}   ← should be lower than baseline")
print(f"NLL      : {metrics_mc['nll']:.4f}")
print(f"Mean entropy IND : {uncertainty_test.mean():.4f}")
print(f"Mean entropy OOD : {uncertainty_ood.mean():.4f}  ← should be higher")

In [ ]:
# ── CELL 7: Temperature Scaling ──────────────────────────────────────────────
# PAPER: Guo 2017 Section 3.3:
#   'We divide logits by scalar T, optimised on the validation set.
#    T > 1 softens overconfident predictions. Accuracy is unchanged.'
# This is the SIMPLEST fix — no retraining, one parameter.
from evaluation import TemperatureScaler, predict_temperature_scaled

ts = TemperatureScaler(model)
T_value = ts.fit(loaders['val'])

probs_ts, _ = predict_temperature_scaled(ts, loaders['test'])
metrics_ts  = compute_all(probs_ts, labels_test)

print('── Temperature Scaling ──')
print(f"Learned T: {T_value:.4f}  (>1 = was overconfident, now softened)")
print(f"Accuracy : {metrics_ts['accuracy']:.4f}  ← same as baseline (unchanged)")
print(f"ECE      : {metrics_ts['ece']:.4f}   ← should be lowest")
print(f"NLL      : {metrics_ts['nll']:.4f}")

In [ ]:
# ── CELL 8: Full results table ───────────────────────────────────────────────
results = {
    'std': {'probs': probs_std, 'labels': labels_test, 'metrics': metrics_std},
    'mc':  {'probs': probs_mc,  'labels': labels_test, 'metrics': metrics_mc},
    'ts':  {'probs': probs_ts,  'labels': labels_test, 'metrics': metrics_ts},
}

print(f"{'='*62}")
print(f"{'CALIBRATION RESULTS':^62}")
print(f"{'='*62}")
print(f"{'Method':<28} {'Accuracy':>9} {'ECE':>9} {'NLL':>9}")
print(f"{'-'*62}")
for key, label in [('std','Standard (Baseline)'),('mc','MC Dropout (T=30)'),('ts','Temperature Scaling')]:
    m = results[key]['metrics']
    print(f"{label:<28} {m['accuracy']:>9.4f} {m['ece']:>9.4f} {m['nll']:>9.4f}")
print(f"{'='*62}")

In [ ]:
# ── CELL 9: Generate all poster figures ─────────────────────────────────────
# Saves 5 PNGs to figures/
# fig1 — Reliability diagrams  ← KEY poster figure
# fig2 — ECE bar chart
# fig3 — Entropy: in-distribution vs OOD
# fig4 — Results table
# fig5 — Training curves
import os
os.makedirs('figures', exist_ok=True)

from visualization import (plot_reliability_diagrams, plot_ece_comparison,
                            plot_entropy_ood, plot_results_table, plot_training_curves)

plot_training_curves(history)
plot_reliability_diagrams(results)
plot_ece_comparison(results)
plot_entropy_ood(uncertainty_test, uncertainty_ood)
plot_results_table(results, dataset_name, T_value)

print('\n✓ All figures saved to figures/')

In [ ]:
# ── CELL 10: Push figures to GitHub ─────────────────────────────────────────
import getpass
token = getpass.getpass('GitHub token: ')

!git config user.email 'inada350@gmail.com'
!git config user.name 'Nada Elhassan'
!git remote set-url origin https://{token}@github.com/nadaibrahim-png/ictp-poster-2026.git
!git add figures/
!git commit -m 'Experiment A: calibration figures'
!git push origin main
print('✓ Figures pushed to GitHub')